# Pepare Data for Accuracy Report

In this notebook, we evaluate the performance of the Language Learning Model (LLM) in answering financial questions. The process involves generating answers using the LLM, comparing these answers to the original human-provided answers, and scoring the LLM's accuracy. The final output is a CSV file containing the original data, LLM-generated answers, and the corresponding scores.

## Imports

In [2]:
import sys
sys.path.append('..')

In [3]:
# %% Imports
import json
import os
import pandas as pd
import time
from dotenv import load_dotenv
from helpers.helper import FinQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

## Functions

In [4]:
def load_data():
    with open("../data/extracted_data.json", "r") as f:
        data = json.load(f)
    return data

## Initialise

In [5]:
# Load the .env file
load_dotenv()

True

In [6]:
data = load_data()
df = pd.DataFrame(data)
print(f"Number of sample data: {len(data)}")

Number of sample data: 3965


In [7]:
gpt_model = "gpt-4o-mini"

In [8]:
fin_qa = FinQA(gpt_model)

c:\Users\Amir.Tavakoli-Kashi\Documents\Personal\tomoro\FinQA-LLM\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:151: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


## Loop Over Samples and get Answers

In [9]:
df_with_llm_answers = df.copy()
df_with_llm_answers["llm_answer"] = None
df_with_llm_answers["responder"] = None

In [10]:
for i, row in df_with_llm_answers.iterrows():
    
    (answer, responder) = fin_qa.get_answer(
        pre_text=row["pre_text"],
        table=row["table"],
        post_text=row["post_text"],
        question=row["question"],
    )

    df_with_llm_answers.loc[i, "llm_answer"] = answer
    df_with_llm_answers.loc[i, "responder"] = responder
    time.sleep(0.5)

c:\Users\Amir.Tavakoli-Kashi\Documents\Personal\tomoro\FinQA-LLM\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:151: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
Thought: To find the percentage change in net cash from operating activities from 2008 to 2009, I will use the formula: 

\[
\text{Percentage Change} = \left( \frac{\text{New Value} - \text{Old Value}}{\text{Old Value}} \right) \times 100
\]

The net cash from operating activities for 2008 is $181001 and for 2009 is $206588. 

Action:
```
{
  "action": "Calculator",
  "action_input": "(206588 - 181001) / 181001 * 100"
}
```

Observation: Answer: 14.136385986817752
Thought:I now know the final answer.
Final Answer: 14.14%

> Finished chain.


> Entering new AgentExecutor chain...
Thought: To calculate the percent growth in revenues from 2007 to 2008, I will use the formula: 

\[
\text{Percent Growth} = \left( \frac{\text{Revenue in 2008} - \text{Revenue in 2007}}{\text{Revenue in 2007}} \right) \times 100
\]

From the table, the revenue for 2008 is $9362.2 million and for 2007 is $9244.9 million. 

Action:
```
{
  "action": "Calculator",
  "action

In [11]:
df_with_llm_answers.head(2).T

,0,1
id,Single_JKHY/2009/page_28.pdf-3,Single_RSG/2008/page_114.pdf-2
pre_text,"[26 | 2009 annual report in fiscal 2008 , reve...",[substantially all of the goodwill and other i...
post_text,"[year ended june 30 , cash provided by operati...",[the above unaudited pro forma financial infor...
filename,JKHY/2009/page_28.pdf,RSG/2008/page_114.pdf
table_ori,"[[, Year ended June 30, 2009], [2008, 2007], [...","[[, Year Ended December 31, 2008 (Unaudited), ..."
table,"[[2008, year ended june 30 2009 2008, year end...","[[, year ended december 31 2008 ( unaudited ),..."
annotation,{'amt_table': '<table class='wikitable'><tr><t...,{'amt_table': '<table class='wikitable'><tr><t...
question,what was the percentage change in the net cash...,what was the percent of the growth in the reve...
answer,14.1%,1.3%
llm_answer,14.14%,1.27%


## Scoring

In [12]:
# Langchain Clients
llm_client = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0,
)

In [13]:
class ScoreResult(BaseModel):
    """Boolean classification of the LLM Results"""

    score: bool = Field(
        description="Are the two inputs the same, 'True' or 'False'"
    )
llm_scoring = llm_client.with_structured_output(ScoreResult)

compare_prompt = ChatPromptTemplate.from_template(
    "You are an expert financial assistant. You are given two responses: "
    "one from a human and one from an LLM. Your task is to determine if "
    "the two responses convey the same meaning, even if the wording or "
    "formatting is slightly different.\n\n"
    "## Guidelines:\n\n"
    "1. Percentages can be rounded (e.g., 27.58% and 27.6% are considered similar).\n"
    "2. Different representations of currency are acceptable as long as they are equivalent (e.g., $3 million and 3,000,000 USD are considered similar).\n"
    "3. Whole numbers and their percentage equivalents are acceptable (e.g., 12% and 12 are considered similar).\n"
    "4. Positive and negative numbers with the same magnitude are not similar (e.g., -12.3% and 12% are not similar).\n\n"
    "### Examples:\n\n"
    "1. Human Response: -53% \n"
    "LLM Response:** -53.86%\n"
    "Are these similar? True\n\n"
    "2. Human Response: $3 million\n"
    "LLM Response:** 3,000,000 USD\n"
    "Are these similar? True\n\n"
    "3. Human Response: 12% \n"
    "LLM Response:** 12\n"
    "Are these similar? True\n\n"
    "4. Human Response: -12.3% \n"
    "LLM Response:** 12%\n"
    "Are these similar? False\n\n"
    "### Input:\n\n"
    "Human Response: {human_response}\n"
    "LLM Response: {llm_response}\n\n"
)

runnable_scoring = compare_prompt | llm_scoring

In [14]:
# Test the runnable
res = runnable_scoring.invoke(
    {
        "human_response":"-32.82%",
        "llm_response":"-32%"
    }
)
res.score

True

In [15]:
df_with_score = df_with_llm_answers.copy()
df_with_score["score"] = None

In [16]:
for i, row in df_with_score.iterrows():
    res = runnable_scoring.invoke(
        {
            "human_response": row["answer"],
            "llm_response": row["llm_answer"]
        }
    )
    df_with_score.loc[i, "score"] = res.score
    time.sleep(0.1)

In [18]:
df_with_score.head(3).T

,0,1,2
id,Single_JKHY/2009/page_28.pdf-3,Single_RSG/2008/page_114.pdf-2,Single_AAPL/2002/page_23.pdf-1
pre_text,"[26 | 2009 annual report in fiscal 2008 , reve...",[substantially all of the goodwill and other i...,[in a new business model such as the retail se...
post_text,"[year ended june 30 , cash provided by operati...",[the above unaudited pro forma financial infor...,[.]
filename,JKHY/2009/page_28.pdf,RSG/2008/page_114.pdf,AAPL/2002/page_23.pdf
table_ori,"[[, Year ended June 30, 2009], [2008, 2007], [...","[[, Year Ended December 31, 2008 (Unaudited), ...","[[, 2002, 2001, 2000], [Net sales, $5,742, $5,..."
table,"[[2008, year ended june 30 2009 2008, year end...","[[, year ended december 31 2008 ( unaudited ),...","[[, 2002, 2001, 2000], [net sales, $ 5742, $ 5..."
annotation,{'amt_table': '<table class='wikitable'><tr><t...,{'amt_table': '<table class='wikitable'><tr><t...,{'amt_table': '<table class='wikitable'><tr><t...
question,what was the percentage change in the net cash...,what was the percent of the growth in the reve...,what was the percentage change in net sales fr...
answer,14.1%,1.3%,-32%
llm_answer,14.14%,1.27%,-32.82%


In [20]:
df_with_score.to_csv("../data/df_with_score.csv", index=False)

In [21]:
df_with_score = pd.read_csv("../data/df_with_score.csv", index_col=False)

In [23]:
df_with_score["score"].value_counts()

score
True     2551
False    1414
Name: count, dtype: int64